# Comparative Evaluation - Spain CPI Forecasting

**Research question:** Do MCP signals (EPU Europe, Brent, BCE/GDELT sentiment) improve IPC Spain forecasts in foundation time series models?

Rolling-origin backtesting 2021–2024 · 23 models · Horizons h=1, 3, 6, 12 months

**Sections:**
1. Which model forecasts best by horizon?
2. Do MCP signals improve over the univariate baseline (C0)?
3. Are there subperiod differences (stability / ECB shock / normalization)?
4. Are the differences statistically significant (Diebold-Mariano test)?
5. **Main figure** for the thesis document
6. Full reference tables

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import warnings
warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
RESULTS = ROOT / '08_results'

metrics = json.load(open(RESULTS / 'metrics_summary_final.json'))
dm_raw  = json.load(open(RESULTS / 'diebold_mariano_results_final.json'))

HORIZONS = [1, 3, 6, 12]
print(f'Modelos: {len(metrics)}  |  Pares DM: {len(dm_raw)}')

Modelos: 23  |  Pares DM: 65


In [ ]:
# ── Color palette by model family ──────────────────────────────────
FAMILY_COLORS = {
    'naive': '#b0b0b0', 'arima': '#888888', 'sarima': '#606060', 'sarimax': '#404040',
    'auto_arima': '#8B4513',
    'lstm': '#4e9af1', 'nbeats': '#1a6ab0', 'nhits': '#0d3d6e',
    'timesfm_C0': '#2ca02c', 'chronos2_C0': '#52a852', 'timegpt_C0': '#8fbc8f',
    'timesfm_C1': '#d62728', 'chronos2_C1': '#ff7f0e', 'timegpt_C1': '#e08080',
    'chronos2_C1_energy': '#e6841a', 'timegpt_C1_energy': '#f5a742',
    'chronos2_C1_energy_only': '#ffc370', 'timegpt_C1_energy_only': '#ffddb0',
    'timesfm_C1_inst': '#9467bd', 'chronos2_C1_inst': '#c5b0d5', 'timegpt_C1_inst': '#b8a0c8',
    'timesfm_C1_macro': '#8c564b', 'chronos2_C1_macro': '#c49c94', 'timegpt_C1_macro': '#d4a89a',
}

MODEL_ORDER = [
    'naive', 'arima', 'sarima', 'sarimax', 'auto_arima',
    'lstm', 'nbeats', 'nhits',
    'timesfm_C0', 'timesfm_C1', 'timesfm_C1_inst', 'timesfm_C1_macro',
    'chronos2_C0', 'chronos2_C1', 'chronos2_C1_energy', 'chronos2_C1_energy_only',
    'chronos2_C1_inst', 'chronos2_C1_macro',
    'timegpt_C0', 'timegpt_C1', 'timegpt_C1_energy', 'timegpt_C1_energy_only',
    'timegpt_C1_inst', 'timegpt_C1_macro',
]
MODEL_ORDER = [m for m in MODEL_ORDER if m in metrics]

# ── Flat metrics DataFrame ─────────────────────────────────────────
rows = []
for model in MODEL_ORDER:
    for h in HORIZONS:
        m = metrics[model].get(f'h{h}', {})
        rows.append({'model': model, 'horizon': h,
                     'MAE': m.get('MAE'), 'RMSE': m.get('RMSE'),
                     'MASE': m.get('MASE'), 'n': m.get('n_evals')})
df = pd.DataFrame(rows)

# ── Flat DM tests DataFrame ────────────────────────────────────────
dm_records = []
for entry in dm_raw:
    for h in HORIZONS:
        hd = entry.get(f'h{h}', {})
        dm_records.append({
            'model1': entry['model1'], 'model2': entry['model2'],
            'period': entry['period'], 'horizon': h,
            'dm_stat': hd.get('dm_stat'), 'p_value': hd.get('p_value'),
            'better': hd.get('better'), 'n': hd.get('n', 0),
        })
dm_df = pd.DataFrame(dm_records)

print(f'df: {df.shape} | dm_df: {dm_df.shape}')
# AutoARIMA comparison
aa = metrics.get('auto_arima', {})
sa = metrics.get('sarima', {})
ar = metrics.get('arima', {})
print('AutoARIMA Spain vs SARIMA/ARIMA:')
for h in HORIZONS:
    mae_aa = aa.get(f'h{h}', {}).get('MAE')
    mae_sa = sa.get(f'h{h}', {}).get('MAE')
    mae_ar = ar.get(f'h{h}', {}).get('MAE')
    if mae_aa:
        print(f'  h={h}: AutoARIMA={mae_aa:.4f}  SARIMA={mae_sa:.4f} ({(mae_aa-mae_sa)/mae_sa*100:+.1f}%)  ARIMA={mae_ar:.4f} ({(mae_aa-mae_ar)/mae_ar*100:+.1f}%)')

---
## 1 · Which model forecasts best by horizon?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7), sharey=False)

sorted_mods = (df[df['horizon']==1].set_index('model')['MAE']
               .reindex(MODEL_ORDER).dropna().sort_values().index.tolist())

for ax, h in zip(axes, HORIZONS):
    sub = (df[(df['horizon']==h) & (df['model'].isin(sorted_mods))]
           .set_index('model').reindex(sorted_mods))
    colors = [FAMILY_COLORS.get(m, '#999') for m in sorted_mods]
    ax.barh(range(len(sorted_mods)), sub['MAE'], color=colors, edgecolor='white', lw=0.4)
    best = sub['MAE'].idxmin()
    ax.barh(sorted_mods.index(best), sub.loc[best,'MAE'],
            color=FAMILY_COLORS.get(best,'#999'), edgecolor='gold', lw=2.5)
    ax.set_yticks(range(len(sorted_mods)))
    ax.set_yticklabels(sorted_mods, fontsize=8)
    ax.set_xlabel('MAE (CPI index points)', fontsize=9)
    ax.set_title(f'h={h}', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

legend_items = [
    mpatches.Patch(color='#888888', label='Statistical'),
    mpatches.Patch(color='#1a6ab0', label='Deep Learning'),
    mpatches.Patch(color='#2ca02c', label='Foundation C0 (univariate)'),
    mpatches.Patch(color='#d62728', label='Foundation C1-MCP (GDELT)'),
    mpatches.Patch(color='#e6841a', label='Foundation C1-Energy (Brent/TTF)'),
    mpatches.Patch(color='#9467bd', label='Foundation C1-Inst (EPU Europe)'),
    mpatches.Patch(color='#8c564b', label='Foundation C1-Macro'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=8,
           frameon=True, bbox_to_anchor=(0.5, -0.04))
fig.suptitle('MAE by model and horizon - Rolling-origin 2021–2024  (gold border = best)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / 'fig1_mae_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

# Ranking table
for h in HORIZONS:
    top = df[df['horizon']==h].sort_values('MAE').head(5)
    print(f'h={h}: ' + ' | '.join(f"{r['model']} {r['MAE']:.4f}" for _, r in top.iterrows()))

---
## 2 · Do MCP signals improve forecasting accuracy? (Relative Δ MAE)

In [ ]:
# ── C1 vs C0 delta heatmap ────────────────────────────────────────
C1_PAIRS = [
    ('TimesFM C1_MCP',     'timesfm_C0',  'timesfm_C1'),
    ('TimesFM C1_inst ★',  'timesfm_C0',  'timesfm_C1_inst'),
    ('TimesFM C1_macro',   'timesfm_C0',  'timesfm_C1_macro'),
    ('Chronos2 C1_MCP',    'chronos2_C0', 'chronos2_C1'),
    ('Chronos2 C1_energy', 'chronos2_C0', 'chronos2_C1_energy'),
    ('Chronos2 C1_eo',     'chronos2_C0', 'chronos2_C1_energy_only'),
    ('Chronos2 C1_inst',   'chronos2_C0', 'chronos2_C1_inst'),
    ('Chronos2 C1_macro',  'chronos2_C0', 'chronos2_C1_macro'),
    ('TimeGPT C1_MCP',     'timegpt_C0',  'timegpt_C1'),
    ('TimeGPT C1_energy',  'timegpt_C0',  'timegpt_C1_energy'),
    ('TimeGPT C1_eo',      'timegpt_C0',  'timegpt_C1_energy_only'),
    ('TimeGPT C1_inst',    'timegpt_C0',  'timegpt_C1_inst'),
    ('TimeGPT C1_macro',   'timegpt_C0',  'timegpt_C1_macro'),
]

hm = np.full((len(C1_PAIRS), 4), np.nan)
for i, (lbl, c0n, c1n) in enumerate(C1_PAIRS):
    for j, h in enumerate(HORIZONS):
        m0 = metrics.get(c0n, {}).get(f'h{h}', {}).get('MAE')
        m1 = metrics.get(c1n, {}).get(f'h{h}', {}).get('MAE')
        if m0 and m1:
            hm[i, j] = (m1 - m0) / m0 * 100

fig, ax = plt.subplots(figsize=(8, 9))
norm = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=65)
im = ax.imshow(hm, cmap='RdYlGn_r', norm=norm, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=11)
ax.set_yticks(range(len(C1_PAIRS)))
ax.set_yticklabels([r[0] for r in C1_PAIRS], fontsize=9)
ax.set_title('Relative Δ MAE: (C1 − C0) / C0 × 100%\n'
             'Green = C1 better  |  Red = C0 better (C1 worse)\n'
             '★ = only C1 that improves globally',
             fontsize=10, fontweight='bold')
for i in range(len(C1_PAIRS)):
    for j in range(4):
        v = hm[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 28 else 'black'
            ax.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                    fontsize=8.5, color=ct, fontweight='bold')
# Family separators
for b in [3, 8]:
    ax.axhline(b - 0.5, color='white', linewidth=2.5)
plt.colorbar(im, ax=ax, shrink=0.65, label='Δ MAE (%)')
plt.tight_layout()
plt.savefig(RESULTS / 'fig2_delta_mae_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── MAE profiles: C0 vs best C1 per family ───────────────────────
PAIRS_PROFILE = [
    ('TimesFM',   'timesfm_C0',  'timesfm_C1_inst',  '#2ca02c', '#9467bd'),
    ('Chronos-2', 'chronos2_C0', 'chronos2_C1_inst', '#52a852', '#c5b0d5'),
    ('TimeGPT',   'timegpt_C0',  'timegpt_C1',       '#8fbc8f', '#e08080'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(HORIZONS))

for ax, (fname, c0, c1, col0, col1) in zip(axes, PAIRS_PROFILE):
    c0v = [metrics[c0][f'h{h}']['MAE'] for h in HORIZONS]
    c1v = [metrics[c1][f'h{h}']['MAE'] for h in HORIZONS]
    ax.plot(x, c0v, 'o-', color=col0, lw=2.5, ms=7, label=c0, zorder=5)
    ax.plot(x, c1v, 's--', color=col1, lw=2.5, ms=7, label=c1, zorder=5)
    ax.fill_between(x, c0v, c1v,
                    where=[v1 < v0 for v0, v1 in zip(c0v, c1v)],
                    alpha=0.18, color='green', label='C1 better')
    ax.fill_between(x, c0v, c1v,
                    where=[v1 >= v0 for v0, v1 in zip(c0v, c1v)],
                    alpha=0.12, color='red', label='C0 better')
    for ref, rlbl, rcol, rls in [
        ('nbeats',     'N-BEATS',          '#888',    ':'),
        ('sarima',     'SARIMA',           '#606060', ':'),
        ('auto_arima', 'AutoARIMA (dyn.)', '#8B4513', '--'),
    ]:
        if ref in metrics:
            rv = [metrics[ref][f'h{h}']['MAE'] for h in HORIZONS]
            ax.plot(x, rv, rls, color=rcol, lw=1.3, alpha=0.70, label=rlbl)
    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}' for h in HORIZONS])
    ax.set_ylabel('MAE', fontsize=9)
    ax.set_title(fname, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.25)

fig.suptitle('C0 (univariate) vs best C1 per family - MAE Profile', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig3_c0_vs_c1_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Subperiod differences

In [ ]:
PERIODS = ['global', 'A_2021', 'B_2022_shock', 'C_2023_2024']
PERIOD_LABELS = {
    'global':       'Global\n2021–2024',
    'A_2021':       'A: Stability\n2021',
    'B_2022_shock': 'B: ECB Shock\n2022',
    'C_2023_2024':  'C: Normalization\n2023–24',
}

PAIRS_SUB = [
    ('timesfm_C0',  'timesfm_C1_inst',  'TimesFM C0 vs C1_inst'),
    ('chronos2_C0', 'chronos2_C1_inst', 'Chronos-2 C0 vs C1_inst'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, (m1, m2, title) in zip(axes, PAIRS_SUB):
    sub = dm_df[(dm_df['model1'] == m1) & (dm_df['model2'] == m2)]
    xp = np.arange(len(PERIODS))
    w  = 0.18
    for i, h in enumerate(HORIZONS):
        dvals, pvals = [], []
        for p in PERIODS:
            row = sub[(sub['period'] == p) & (sub['horizon'] == h)]
            if len(row) and pd.notna(row['dm_stat'].values[0]):
                dvals.append(float(row['dm_stat'].values[0]))
                pvals.append(float(row['p_value'].values[0]))
            else:
                dvals.append(0.0); pvals.append(1.0)
        ax.bar(xp + (i - 1.5) * w, dvals, w, label=f'h={h}',
               alpha=0.85, color=plt.cm.plasma(i / 4))
        for xi, (dv, pv) in enumerate(zip(dvals, pvals)):
            if pv < 0.05:
                ax.text(xi + (i - 1.5) * w, dv + (0.22 if dv >= 0 else -0.48),
                        '**', ha='center', fontsize=9, fontweight='bold')
            elif pv < 0.10:
                ax.text(xi + (i - 1.5) * w, dv + (0.22 if dv >= 0 else -0.42),
                        '*', ha='center', fontsize=9)

    ax.axhline(0,    color='black', lw=1)
    ax.axhline(-1.96, color='red',   lw=0.9, ls='--', alpha=0.55, label='α=5% (C0 better)')
    ax.axhline( 1.96, color='green', lw=0.9, ls='--', alpha=0.55, label='α=5% (C1 better)')
    ax.set_xticks(xp)
    ax.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS], fontsize=9)
    ax.set_ylabel('DM statistic', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7.5, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.25)
    ax.set_ylim(-7, 7)
    ax.annotate('DM > 0 → C1 better\nDM < 0 → C0 better',
                xy=(0.02, 0.97), xycoords='axes fraction', fontsize=7,
                va='top', color='gray', bbox=dict(boxstyle='round', fc='white', alpha=0.7))

fig.suptitle('Diebold-Mariano test by subperiod: C0 vs C1_inst (EPU Europe)\n'
             '** p<0.05  * p<0.10', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / 'fig4_dm_subperiod_inst.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3b · Month-by-month absolute error over time

Absolute error h=1 over 2021–2024. Shaded backgrounds mark the three regimes:
**A** - pre-shock stability (2021), **B** - inflation shock / ECB rate hikes (2022),
**C** - post-shock normalization (2023–2024).

N-BEATS has quarterly origins (16 points); all others are monthly (47 pts).

In [ ]:
# ── Fig 3b: Temporal absolute error h=1 (2021–2024) ─────────────────
import pandas as pd

BASELINE_PREDS = ROOT / '03_models_baseline' / 'results' / 'rolling_predictions.parquet'
DEEP_PREDS     = ROOT / '04_models_deep'     / 'results' / 'deep_rolling_predictions.parquet'

_df_base = pd.read_parquet(BASELINE_PREDS)
_df_base['origin']  = pd.to_datetime(_df_base['origin'])
_df_base['fc_date'] = pd.to_datetime(_df_base['fc_date'])

_df_deep = pd.read_parquet(DEEP_PREDS)
_df_deep['origin']  = pd.to_datetime(_df_deep['origin'])
_df_deep['fc_date'] = pd.to_datetime(_df_deep['fc_date'])

def _load_foundation(name):
    d = pd.read_parquet(RESULTS / f'{name}_predictions.parquet')
    d['origin']  = pd.to_datetime(d['origin'])
    d['fc_date'] = pd.to_datetime(d['fc_date'])
    return d

def _get_h1_errors(name):
    if name == 'nbeats':
        sub = _df_deep[(_df_deep['model']=='nbeats') & (_df_deep['horizon']==1) & (_df_deep['step']==1)]
    elif name in ('sarima', 'arima', 'naive', 'sarimax'):
        sub = _df_base[(_df_base['model']==name) & (_df_base['horizon']==1) & (_df_base['step']==1)]
    else:
        d = _load_foundation(name)
        sub = d[(d['horizon']==1) & (d['step']==1)]
    return sub.set_index('fc_date')['abs_error'].sort_index()

# Subperiod shading (reused in fig 3c too)
SHADING = [
    ('A: Stability',      '2021-01-01', '2021-12-31', '#d4e8d4', 0.45),
    ('B: ECB Shock',      '2022-01-01', '2022-12-31', '#f8d7d7', 0.50),
    ('C: Normalization',  '2023-01-01', '2024-12-31', '#d7e8f8', 0.35),
]

# Models: (name, color, label, linestyle, linewidth, marker)
TIMELINE_MODELS = [
    ('sarima',          '#606060', 'SARIMA',              '-',  1.5, 'o'),
    ('nbeats',          '#1a6ab0', 'N-BEATS (trim.)',     '--', 1.8, 'D'),
    ('timesfm_C0',      '#2ca02c', 'TimesFM C0',          '-',  2.0, 's'),
    ('chronos2_C0',     '#52a852', 'Chronos-2 C0',        '-',  1.8, '^'),
    ('timesfm_C1_inst', '#9467bd', 'TimesFM C1_inst ★',   '-',  2.2, 'v'),
]

fig, ax = plt.subplots(figsize=(14, 5))

# Background shading
for lbl, s, e, col, alpha in SHADING:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=alpha, zorder=0)

# Plot each model
for name, col, label, ls, lw, mk in TIMELINE_MODELS:
    try:
        series = _get_h1_errors(name)
    except Exception as ex:
        print(f'[!] {name}: {ex}'); continue
    if name == 'nbeats':
        # Quarterly origins: scatter + dotted connector (no interpolation)
        ax.scatter(series.index, series.values, color=col, marker=mk, s=60, zorder=6, label=label)
        ax.plot(series.index, series.values, ':', color=col, lw=1.0, alpha=0.45, zorder=5)
    else:
        ax.plot(series.index, series.values, ls=ls, color=col, lw=lw,
                marker=mk, markersize=4, markevery=4, label=label, zorder=5)

# Period labels (drawn after axes limits are set)
ylim = ax.get_ylim()
for lbl, s, e, col, alpha in SHADING:
    ax.text(pd.Timestamp(s) + pd.Timedelta(days=22), ylim[1] * 0.965,
            lbl, fontsize=8.5, color='#333', va='top', fontweight='bold')

ax.set_ylabel('Absolute error h=1 (CPI index points)', fontsize=10)
ax.set_xlabel('Forecast date', fontsize=10)
ax.set_title(
    'Month-by-month absolute error - Horizon h=1 (2021–2024)\n'
    'Monthly origins (line) · N-BEATS: quarterly origins (scatter + dotted)',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9, loc='upper left', framealpha=0.85)
ax.grid(alpha=0.22, zorder=0)
ax.set_xlim(pd.Timestamp('2021-01-01'), pd.Timestamp('2025-01-01'))

plt.tight_layout()
plt.savefig(RESULTS / 'fig3b_error_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3b_error_temporal.png')

In [ ]:
# ── Fig 3c: Forecast fan Chronos-2 C0 (p10–p90 intervals) ────────────
# Chronos-2 is the only model in this TFG with probabilistic output.
# 80% prediction interval (p10, p90) over observed CPI at h=1.

df_ch = pd.read_parquet(RESULTS / 'chronos2_C0_predictions.parquet')
df_ch['fc_date'] = pd.to_datetime(df_ch['fc_date'])
df_ch['origin']  = pd.to_datetime(df_ch['origin'])

h1_ch = df_ch[(df_ch['horizon']==1) & (df_ch['step']==1)].sort_values('fc_date').copy()

within_80 = ((h1_ch['y_true'] >= h1_ch['y_pred_p10']) &
             (h1_ch['y_true'] <= h1_ch['y_pred_p90'])).mean()
ic_width  = (h1_ch['y_pred_p90'] - h1_ch['y_pred_p10']).mean()
print(f'Empirical coverage 80% PI (p10–p90): {within_80:.1%}  (nominal: 80%)')
print(f'Mean PI width: {ic_width:.3f} index points')

fig, ax = plt.subplots(figsize=(14, 5))

# Subperiod shading
for lbl, s, e, col, alpha in SHADING:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color=col, alpha=alpha, zorder=0)

# IC band (80%)
ax.fill_between(h1_ch['fc_date'], h1_ch['y_pred_p10'], h1_ch['y_pred_p90'],
                color='#52a852', alpha=0.22, label=f'80% PI (p10–p90)', zorder=2)
# IC border lines
ax.plot(h1_ch['fc_date'], h1_ch['y_pred_p10'], '--', color='#52a852', lw=0.9, alpha=0.6, zorder=3)
ax.plot(h1_ch['fc_date'], h1_ch['y_pred_p90'], '--', color='#52a852', lw=0.9, alpha=0.6, zorder=3)

# Median prediction
ax.plot(h1_ch['fc_date'], h1_ch['y_pred'], '-', color='#2ca02c', lw=2.2,
        label='Chronos-2 C0 (median)', zorder=5)

# Observed CPI
ax.plot(h1_ch['fc_date'], h1_ch['y_true'], 'k-', lw=1.8, label='Observed CPI', zorder=6)
ax.scatter(h1_ch['fc_date'], h1_ch['y_true'], c='black', s=20, zorder=7)

# Points outside the PI
outside = h1_ch[(h1_ch['y_true'] < h1_ch['y_pred_p10']) | (h1_ch['y_true'] > h1_ch['y_pred_p90'])]
if len(outside) > 0:
    ax.scatter(outside['fc_date'], outside['y_true'], c='#cc0000', s=80, zorder=8,
               marker='x', linewidths=2.2, label=f'Outside 80% PI ({len(outside)} months)')

# Period labels
ylim = ax.get_ylim()
for lbl, s, e, col, alpha in SHADING:
    ax.text(pd.Timestamp(s) + pd.Timedelta(days=22), ylim[0] + (ylim[1]-ylim[0])*0.04,
            lbl, fontsize=8.5, color='#333', fontweight='bold')

ax.set_ylabel('Spain CPI Index (base 2021=100)', fontsize=10)
ax.set_xlabel('Forecast date', fontsize=10)
ax.set_title(
    f'Forecast Fan - Chronos-2 C0 · Horizon h=1 · 80% Prediction Interval (p10–p90)\n'
    f'Empirical coverage: {within_80:.1%} (nominal 80%) · '
    f'Mean PI width: {ic_width:.2f} index points',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9, loc='upper left', framealpha=0.88)
ax.grid(alpha=0.22, zorder=0)
ax.set_xlim(pd.Timestamp('2021-01-01'), pd.Timestamp('2025-01-01'))

plt.tight_layout()
plt.savefig(RESULTS / 'fig3c_forecast_fan.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3c_forecast_fan.png')

---
## 4 · Statistical Significance (DM test - global C0 vs all C1)

In [ ]:
# ── DM statistic heatmap ──────────────────────────────────────────
C1_DM = [
    ('timesfm_C0',  'timesfm_C1'),
    ('timesfm_C0',  'timesfm_C1_inst'),
    ('timesfm_C0',  'timesfm_C1_macro'),
    ('chronos2_C0', 'chronos2_C1'),
    ('chronos2_C0', 'chronos2_C1_energy'),
    ('chronos2_C0', 'chronos2_C1_energy_only'),
    ('chronos2_C0', 'chronos2_C1_inst'),
    ('chronos2_C0', 'chronos2_C1_macro'),
    ('timegpt_C0',  'timegpt_C1'),
    ('timegpt_C0',  'timegpt_C1_energy'),
    ('timegpt_C0',  'timegpt_C1_energy_only'),
    ('timegpt_C0',  'timegpt_C1_inst'),
    ('timegpt_C0',  'timegpt_C1_macro'),
]

dm_hm = np.full((len(C1_DM), 4), np.nan)
dm_pv = np.full((len(C1_DM), 4), np.nan)
for i, (m1, m2) in enumerate(C1_DM):
    sub = dm_df[(dm_df['model1']==m1) & (dm_df['model2']==m2) & (dm_df['period']=='global')]
    for j, h in enumerate(HORIZONS):
        hr = sub[sub['horizon']==h]
        if len(hr) and pd.notna(hr['dm_stat'].values[0]):
            dm_hm[i, j] = float(hr['dm_stat'].values[0])
            dm_pv[i, j] = float(hr['p_value'].values[0])

fig, ax = plt.subplots(figsize=(8, 7.5))
norm2 = TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)
im2 = ax.imshow(dm_hm, cmap='RdYlGn', norm=norm2, aspect='auto')
ax.set_xticks(range(4))
ax.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=10)
ax.set_yticks(range(len(C1_DM)))
ax.set_yticklabels([m2 for _, m2 in C1_DM], fontsize=8.5)
ax.set_title('DM statistic: C0 vs C1 - global period\n'
             'Green = C1 better  |  Red = C0 better  |  ** p<0.05  * p<0.10',
             fontsize=10, fontweight='bold')
for i in range(len(C1_DM)):
    for j in range(4):
        if not np.isnan(dm_hm[i, j]):
            sig = '**' if dm_pv[i,j] < 0.05 else ('*' if dm_pv[i,j] < 0.10 else '')
            ct  = 'white' if abs(dm_hm[i,j]) > 2.8 else 'black'
            ax.text(j, i, f'{dm_hm[i,j]:.1f}{sig}', ha='center', va='center',
                    fontsize=8.5, color=ct)
for b in [3, 8]:
    ax.axhline(b - 0.5, color='white', lw=2.5)
plt.colorbar(im2, ax=ax, shrink=0.72, label='DM statistic')
plt.tight_layout()
plt.savefig(RESULTS / 'fig5_dm_global_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# All vs naive
print('\n=== Do all models beat Naive? (DM global p<0.05) ===')
naive_p = dm_df[(dm_df['model1']=='naive') & (dm_df['period']=='global')]
for m2 in sorted(naive_p['model2'].unique()):
    sub = naive_p[naive_p['model2']==m2]
    checks = []
    for h in HORIZONS:
        hr = sub[sub['horizon']==h]
        if len(hr) and pd.notna(hr['p_value'].values[0]):
            checks.append('✓' if float(hr['p_value'].values[0]) < 0.05 else '-')
        else:
            checks.append('?')
    print(f'  {m2:<35} ' + '  '.join(f'h={h}:{c}' for h,c in zip(HORIZONS, checks)))

---
## 5 · Main Figure - For the Thesis Document

2×2 panel: (A) MAE ranking h=1, (B) MAE profiles key models, (C) delta heatmap, (D) DM subperiod TimesFM

In [ ]:
matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

fig = plt.figure(figsize=(17, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.44, wspace=0.34)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])

# ── A: MAE Ranking h=1 (top 12 excl. naive) ──────────────────────
df_h1_top = (df[(df['horizon']==1) & (df['model']!='naive')]
             .sort_values('MAE').head(12))
mods_a  = df_h1_top['model'].tolist()
cols_a  = [FAMILY_COLORS.get(m, '#999') for m in mods_a]
ax_a.barh(range(len(mods_a)), df_h1_top['MAE'].values,
          color=cols_a, edgecolor='white', lw=0.5)
ax_a.barh(0, df_h1_top['MAE'].iloc[0], color=cols_a[0], edgecolor='gold', lw=2.5)
if 'timesfm_C1_inst' in mods_a:
    idx_i = mods_a.index('timesfm_C1_inst')
    ax_a.barh(idx_i, df_h1_top.iloc[idx_i]['MAE'],
              color='#9467bd', edgecolor='#3b1a7a', lw=2.2, zorder=6)
ax_a.set_yticks(range(len(mods_a)))
ax_a.set_yticklabels(mods_a, fontsize=8.5)
ax_a.set_xlabel('MAE (CPI index points)', fontsize=9)
ax_a.set_title('(A) MAE Ranking - h=1', fontsize=10, fontweight='bold')
ax_a.grid(axis='x', alpha=0.22)
ax_a.invert_yaxis()

# ── B: MAE profiles key models ────────────────────────────────────
PROFILE = [
    ('nbeats',          '#1a6ab0', 'Deep: N-BEATS',                'o-',  2.2),
    ('sarima',          '#606060', 'Statistical: SARIMA',           's:',  1.6),
    ('auto_arima',      '#8B4513', 'Statistical: AutoARIMA (dyn.)', 'x--', 1.6),
    ('timesfm_C0',      '#2ca02c', 'Foundation C0 (TimesFM)',       'D-',  2.0),
    ('timesfm_C1_inst', '#9467bd', 'Foundation C1_inst (EPU Eu.)',  'v--', 2.4),
    ('chronos2_C0',     '#52a852', 'Foundation C0 (Chronos-2)',     'P-',  1.8),
    ('timegpt_C0',      '#8fbc8f', 'Foundation C0 (TimeGPT)',       '^-',  1.8),
]
xb = np.arange(len(HORIZONS))
for mn, col, lbl, sty, lw in PROFILE:
    if mn not in metrics: continue
    vals = [metrics[mn].get(f'h{h}', {}).get('MAE') for h in HORIZONS]
    ax_b.plot(xb, vals, sty, color=col, lw=lw, ms=7, label=lbl, zorder=5)
ax_b.set_xticks(xb)
ax_b.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_b.set_ylabel('MAE', fontsize=9)
ax_b.set_title('(B) MAE Profile - Key Models', fontsize=10, fontweight='bold')
ax_b.legend(fontsize=7, loc='upper left', framealpha=0.75)
ax_b.grid(alpha=0.22)

# ── C: Compact delta heatmap ──────────────────────────────────────
HM_C = [
    ('TimesFM C1_MCP',    'timesfm_C0',  'timesfm_C1'),
    ('TimesFM C1_inst ★', 'timesfm_C0',  'timesfm_C1_inst'),
    ('TimesFM C1_macro',  'timesfm_C0',  'timesfm_C1_macro'),
    ('Chronos2 C1_MCP',   'chronos2_C0', 'chronos2_C1'),
    ('Chronos2 C1_inst',  'chronos2_C0', 'chronos2_C1_inst'),
    ('Chronos2 C1_eo',    'chronos2_C0', 'chronos2_C1_energy_only'),
    ('TimeGPT C1_MCP',    'timegpt_C0',  'timegpt_C1'),
    ('TimeGPT C1_inst',   'timegpt_C0',  'timegpt_C1_inst'),
    ('TimeGPT C1_energy', 'timegpt_C0',  'timegpt_C1_energy'),
]
hmc = np.full((len(HM_C), 4), np.nan)
for i, (_, c0n, c1n) in enumerate(HM_C):
    for j, h in enumerate(HORIZONS):
        m0 = metrics.get(c0n, {}).get(f'h{h}', {}).get('MAE')
        m1 = metrics.get(c1n, {}).get(f'h{h}', {}).get('MAE')
        if m0 and m1: hmc[i, j] = (m1 - m0) / m0 * 100

norm_c = TwoSlopeNorm(vmin=-15, vcenter=0, vmax=60)
imc = ax_c.imshow(hmc, cmap='RdYlGn_r', norm=norm_c, aspect='auto')
ax_c.set_xticks(range(4))
ax_c.set_xticklabels([f'h={h}' for h in HORIZONS], fontsize=9)
ax_c.set_yticks(range(len(HM_C)))
ax_c.set_yticklabels([r[0] for r in HM_C], fontsize=8.5)
ax_c.set_title('(C) Δ MAE: (C1−C0)/C0×100%\nGreen=improvement · Red=worse',
               fontsize=10, fontweight='bold')
for i in range(len(HM_C)):
    for j in range(4):
        v = hmc[i, j]
        if not np.isnan(v):
            ct = 'white' if abs(v) > 22 else 'black'
            ax_c.text(j, i, f'{v:+.1f}%', ha='center', va='center',
                      fontsize=8, color=ct, fontweight='bold')
for b in [3, 6]:
    ax_c.axhline(b - 0.5, color='white', lw=2.5)
plt.colorbar(imc, ax=ax_c, shrink=0.78, label='Δ MAE (%)')

# ── D: DM subperiod TimesFM C0 vs C1_inst ─────────────────────────
sub_d = dm_df[(dm_df['model1']=='timesfm_C0') & (dm_df['model2']=='timesfm_C1_inst')]
xd = np.arange(len(PERIODS))
wd = 0.18
for i, h in enumerate(HORIZONS):
    dvals, pvals = [], []
    for p in PERIODS:
        row = sub_d[(sub_d['period']==p) & (sub_d['horizon']==h)]
        if len(row) and pd.notna(row['dm_stat'].values[0]):
            dvals.append(float(row['dm_stat'].values[0]))
            pvals.append(float(row['p_value'].values[0]))
        else:
            dvals.append(0.0); pvals.append(1.0)
    ax_d.bar(xd + (i-1.5)*wd, dvals, wd, label=f'h={h}',
             alpha=0.85, color=plt.cm.viridis(i/4), zorder=4)
    for xi, (dv, pv) in enumerate(zip(dvals, pvals)):
        if pv < 0.05:
            ax_d.text(xi+(i-1.5)*wd, dv+(0.22 if dv>=0 else -0.52),
                      '**', ha='center', fontsize=9, fontweight='bold')
        elif pv < 0.10:
            ax_d.text(xi+(i-1.5)*wd, dv+(0.22 if dv>=0 else -0.45),
                      '*', ha='center', fontsize=9)

ax_d.axhline(0,     color='black',   lw=1,   zorder=5)
ax_d.axhline(-1.96, color='#cc0000', lw=0.9, ls='--', alpha=0.6)
ax_d.axhline( 1.96, color='#006600', lw=0.9, ls='--', alpha=0.6)
ax_d.text(3.62, -1.96, 'α=5%', fontsize=7, color='#cc0000', va='center')
ax_d.text(3.62,  1.96, 'α=5%', fontsize=7, color='#006600', va='center')
ax_d.set_xticks(xd)
ax_d.set_xticklabels([PERIOD_LABELS[p] for p in PERIODS], fontsize=8.5)
ax_d.set_ylabel('DM statistic', fontsize=9)
ax_d.set_title('(D) DM: TimesFM C0 vs C1_inst (EPU Europe)\n'
               'DM>0 → C1_inst better  |  ** p<0.05  * p<0.10',
               fontsize=10, fontweight='bold')
ax_d.legend(fontsize=8, ncol=2)
ax_d.grid(axis='y', alpha=0.22, zorder=0)
ax_d.set_ylim(-6.5, 6.5)

# ── Global title and legend ───────────────────────────────────────
fig.suptitle(
    'Comparative Evaluation - Spain CPI Forecasting · Rolling-origin 2021–2024\n'
    'MCP Signals: GDELT (ECB tone/discourse), EPU Europe, Brent/TTF · Metric: MAE',
    fontsize=12, fontweight='bold', y=1.01
)
legend_patches = [
    mpatches.Patch(color='#888888', label='Statistical (ARIMA/SARIMA)'),
    mpatches.Patch(color='#8B4513', label='Statistical AutoARIMA (dynamic)'),
    mpatches.Patch(color='#1a6ab0', label='Deep Learning (N-BEATS/N-HiTS/LSTM)'),
    mpatches.Patch(color='#2ca02c', label='Foundation C0 (univariate)'),
    mpatches.Patch(color='#d62728', label='Foundation C1-MCP (GDELT)'),
    mpatches.Patch(color='#e6841a', label='Foundation C1-Energy (Brent/TTF)'),
    mpatches.Patch(color='#9467bd', label='Foundation C1-Inst (EPU Europe) ★ only C1 that improves'),
    mpatches.Patch(color='#8c564b', label='Foundation C1-Macro (mix Brent+TTF+EPU)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=7.5,
           frameon=True, bbox_to_anchor=(0.5, -0.065),
           title='Model families', title_fontsize=9)

plt.savefig(RESULTS / 'fig_MAIN_summary.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('\n=== MAIN FIGURE: 08_results/fig_MAIN_summary.png ===')

---
## 6 · Complete Reference Tables

In [11]:
pivot_mae  = df.pivot(index='model', columns='horizon', values='MAE').reindex(MODEL_ORDER)
pivot_mase = df.pivot(index='model', columns='horizon', values='MASE').reindex(MODEL_ORDER)
pivot_mae.columns  = [f'h={h}' for h in pivot_mae.columns]
pivot_mase.columns = [f'h={h}' for h in pivot_mase.columns]

print('=== MAE (CPI index points) ===')
print(pivot_mae.round(4).to_string())
print()
print('=== MASE (scale: naive lag-12 on 2002–2020 | MASE < 1 = better than naive) ===')
print(pivot_mase.round(4).to_string())

=== MAE (puntos índice IPC) ===
                            h=1     h=3     h=6    h=12
model                                                  
naive                    3.6259  3.6558  3.6644  6.5881
arima                    0.4781  0.6716  0.9660  1.5410
sarima                   0.4418  0.7236  1.0083  1.5952
sarimax                  0.4472  0.7321  1.0240  1.6070
lstm                     0.3767  1.0922  1.5618  2.3176
nbeats                   0.3588  0.6703  1.0993  1.8947
nhits                    0.4369  0.7935  1.2116  1.7832
timesfm_C0               0.4364  0.7320  1.0866  1.8635
timesfm_C1               0.6811  1.0144  1.4468  2.5001
timesfm_C1_inst          0.4229  0.7056  1.0460  1.8157
timesfm_C1_macro         0.6017  0.8376  1.1869  1.9875
chronos2_C0              0.5202  0.8179  1.2060  1.9898
chronos2_C1              0.5687  0.8605  1.3498  2.3880
chronos2_C1_energy       0.6561  0.9276  1.2720  2.0315
chronos2_C1_energy_only  0.6007  0.8659  1.2148  1.9233
chronos2_C1_inst

In [ ]:
print('=' * 72)
print('RESULTS SUMMARY - TFG IPC-MCP')
print('=' * 72)

aa = metrics.get('auto_arima', {})
sa = metrics.get('sarima',     {})
ar = metrics.get('arima',      {})

print("""
BEST MODEL BY HORIZON (MAE):
  h=1:  N-BEATS  (deep)   MAE=0.359  MASE=0.255
  h=3:  N-BEATS  (deep)   MAE=0.670  MASE=0.477
  h=6:  ARIMA    (stat.)  MAE=0.966  MASE=0.687
  h=12: ARIMA    (stat.)  MAE=1.541  MASE=1.097
""")

print('AUTOARIMA (dynamic order selection) - METHODOLOGICAL FINDING:')
print('  auto_arima reselects SARIMA(p,d,q)(P,D,Q,12) at each of the 48 rolling origins.')
for h in HORIZONS:
    mae_aa = aa.get(f'h{h}', {}).get('MAE')
    mae_sa = sa.get(f'h{h}', {}).get('MAE')
    mae_ar = ar.get(f'h{h}', {}).get('MAE')
    if mae_aa and mae_sa and mae_ar:
        print(f'  h={h:2d}: AutoARIMA={mae_aa:.4f}  vs SARIMA={mae_sa:.4f} ({(mae_aa-mae_sa)/mae_sa*100:+.1f}%)'
              f'  vs ARIMA={mae_ar:.4f} ({(mae_aa-mae_ar)/mae_ar*100:+.1f}%)')
print("""  → Dynamic reselection WORSENS vs fixed orders across all horizons.
    Cause: auto_arima overfits to the recent subsample of each rolling window;
    fixed orders (selected once on 2002–2020) generalize better.
    Implication: for Spain CPI, calibrating once on the full history
    outperforms automatic reselection at each rolling origin.
""")

print("""MCP SIGNALS - CENTRAL FINDING:
  ★ timesfm_C1_inst (EPU Europe): ONLY C1 that improves over C0
    h=1: -3.1% | h=3: -3.6% | h=6: -3.7% | h=12: -2.6%
    → Not globally significant at 5% (p~0.09 at h=6,12)
    → Significant in subperiods: 2021 (**) and Shock 2022 (**)
  ✗ C1-MCP standard (GDELT): worsens +33% to +57% across all families
  ✗ TimeGPT: any exogenous signal significantly degrades forecasts
  ✗ Chronos-2: no C1 condition improves consistently

MOST INFORMATIVE SIGNAL: EPU Europe
  corr(level, CPI_t+1) = +0.737  |  corr(diff, ΔCPI_t+1) = -0.09
  → Captures the price regime, not month-to-month variations

ALL MODELS BEAT THE NAIVE (DM p<0.001 across all horizons)
""")
print('=' * 72)